# Pyspark Assignment

1. Spark Application Setup
    The application must initialize a valid environment:

    Create a SparkSession with a clear application name.

    Explicitly configure the master URL to local mode.


In [40]:
from pyspark.sql import SparkSession

try:
    print('step-1 started...')
    # creating spark session
    spark = SparkSession.builder\
                        .appName('ECommerceAnalyticsPipeline')\
                        .getOrCreate()

    # explicitly configuring master URL
    spark.conf.set('spark.master','local[*]')

    print('step-1 finished successfully')
except Exception as e:
    print(f'Error in creating spark session {e}')
finally:
    print(f'Step-1 over')

step-1 started...
step-1 finished successfully
Step-1 over


### dependecies

In [41]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import lower,upper,col,trim,initcap,round,desc


2. Data Ingestion & Validation
    Validation Logic: Separate valid records from invalid/malformed records.

    Audit: Report the total number of records vs. incomplete records.

    Schema: Enforce an explicit schema (StructType) during the ingestion of valid data.


In [42]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import col

try:
    print('Step-2 started')

    print('\n\n===================== PRODUCTS =====================')
    print('Ingesting Products...')

    product_schema = StructType([
        StructField('product_id', StringType(), False),
        StructField('product_name', StringType(), True),
        StructField('brand', StringType(), True),
        StructField('corrupted_record', StringType(), True)
    ])

    products_df = spark.read\
        .option('header', True)\
        .option('mode', 'PERMISSIVE')\
        .option('columnNameOfCorruptRecord', 'corrupted_record')\
        .schema(product_schema)\
        .csv('products.csv')

    total_products = products_df.count()

    invalid_products_df = products_df.filter(
        (col("product_id").isNull()) |
        (col("product_id") == "") |
        (col("corrupted_record").isNotNull())
    )

    valid_products_df = products_df.filter(
        (col("product_id").isNotNull()) &
        (col("product_id") != "") &
        (col("corrupted_record").isNull())
    ).drop("corrupted_record")

    invalid_products_count = invalid_products_df.count()

    print(f'Total products: {total_products}, Invalid products: {invalid_products_count}')
    print('Invalid products:')
    invalid_products_df.show()

    print('Valid products:')
    valid_products_df.show()


    print('\n\n===================== TRANSACTIONS =====================')
    print('Ingesting Transactions...')

    transaction_schema = StructType([
        StructField('transaction_id', StringType(), False),
        StructField('user_id', StringType(), True),
        StructField('product_id', StringType(), True),
        StructField('category', StringType(), True),
        StructField('price', DoubleType(), True),
        StructField('quantity', IntegerType(), True),
        StructField('transaction_timestamp', TimestampType(), True),
        StructField('platform', StringType(), True),
        StructField('country', StringType(), True),
        StructField('corrupted_record', StringType(), True)
    ])

    transactions_df = spark.read\
        .option('header', True)\
        .option('mode', 'PERMISSIVE')\
        .option('columnNameOfCorruptRecord', 'corrupted_record')\
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")\
        .schema(transaction_schema)\
        .csv('transactions.csv')

    total_transactions = transactions_df.count()

    invalid_transactions_df = transactions_df.filter(
        (col("transaction_id").isNull()) |
        (col("transaction_id") == "") |
        (col("product_id").isNull()) |
        (col("product_id") == "") |
        (col("user_id").isNull()) |
        (col("user_id") == "") |
        (col("quantity").isNull()) |
        (col("price").isNull()) |
        (col("corrupted_record").isNotNull())
    )

    valid_transactions_df = transactions_df.filter(
        (col("transaction_id").isNotNull()) &
        (col("transaction_id") != "") &
        (col("product_id").isNotNull()) &
        (col("product_id") != "") &
        (col("user_id").isNotNull()) &
        (col("user_id") != "") &
        (col("quantity").isNotNull()) &
        (col("price").isNotNull()) &
        (col("corrupted_record").isNull())
    ).drop("corrupted_record")

    invalid_transactions_count = invalid_transactions_df.count()

    print(f'Total transactions: {total_transactions}, Invalid transactions: {invalid_transactions_count}')

    print('Invalid transactions:')
    invalid_transactions_df.show()

    print('Valid transactions:')
    valid_transactions_df.show()


    print('Step-2 finished successfully')

except Exception as e:
    print(f'Error in step-2(Data ingestion and validation): {e}')

finally:
    print('Step-2 over')

Step-2 started


===================== PRODUCTS =====================
Ingesting Products...
Total products: 100, Invalid products: 0
Invalid products:
+----------+------------+-----+----------------+
|product_id|product_name|brand|corrupted_record|
+----------+------------+-----+----------------+
+----------+------------+-----+----------------+

Valid products:
+----------+--------------------+-----------+
|product_id|        product_name|      brand|
+----------+--------------------+-----------+
|   PROD001|Wireless Noise Ca...|  TechSound|
|   PROD002|   Bluetooth Speaker|   AudioMax|
|   PROD003|Designer Leather ...| FashionHub|
|   PROD004|     Garden Tool Set|  GreenLife|
|   PROD005|Python Programmin...| BookMaster|
|   PROD006| 4K Smart TV 55 inch| VisionTech|
|   PROD007|Casual Cotton T-S...|  StyleWear|
|   PROD008|       Gaming Laptop|    TechPro|
|   PROD009|Data Science Hand...| LearnPress|
|   PROD010|Indoor Plant Pot Set|  HomeDecor|
|   PROD011|   Tennis Racket Pro|   Sp

### 3. Data Cleaning & Standardization
    Apply transformations to ensure data readiness:

    Null Handling: Remove or impute missing values.

    Deduplication: Identify and remove duplicate transaction records.

    Standardization: Normalize inconsistent category values and cast columns to correct numeric/timestamp types.



In [43]:
from pyspark.sql.functions import lower,upper,col,trim,initcap

try:
    print('step-3 started')

    # Null Handling of transaction_id, product_id, user_id, quantity, price are handled in previous step (step-2)
    # handling of those field are done by removing records if one of these field is NULL
    # 
    # Now for transaction_timestamp we keep it as it is so that we can keep original data without data loss, 
    # we can later filter records for time based analytics 
    #
    # for optional fields we will replace them with 'unknown'

    valid_transactions_df = valid_transactions_df.fillna({
        "country": "Unknown",
        "platform": "Unknown",
        "category": "Unknown"
    })

    # Standardization
    valid_transactions_df = valid_transactions_df.withColumn("category", lower(trim(col("category"))))
    valid_transactions_df = valid_transactions_df.withColumn("country", upper(trim(col("country"))))
    valid_transactions_df = valid_transactions_df.withColumn("platform", initcap(trim(col("platform"))))    

    # deDuplication

    total_transactions = valid_transactions_df.count()
    print(f'Total transactions before deduplication: {total_transactions}')

    valid_transactions_df = valid_transactions_df.dropDuplicates(["transaction_id"])
    valid_transactions_df.cache()

    valid_transactions_df.show()

    total_transactions = valid_transactions_df.count()
    print(f'Total transactions after deduplication: {total_transactions}')
    print('step-3 finished successfully!')
except Exception as e: 
    print(f'Error in step-3(Data cleaning & standardization) {e}')
finally: 
    print('Step-3 over')

step-3 started
Total transactions before deduplication: 201
+--------------+--------+----------+-------------+-------+--------+---------------------+--------+-------+
|transaction_id| user_id|product_id|     category|  price|quantity|transaction_timestamp|platform|country|
+--------------+--------+----------+-------------+-------+--------+---------------------+--------+-------+
|        TXN087|USER1087|   PROD087|      fashion|  115.0|       1|  2024-01-23 14:22:35|  Mobile| FRANCE|
|        TXN151|USER1151|   PROD051|       sports|  355.0|       1|  2024-01-30 08:30:45|  Mobile|    USA|
|        TXN017|USER1017|   PROD017|home & garden|  89.99|       4|  2024-01-16 14:10:55|  Mobile|GERMANY|
|        TXN101|USER1101|   PROD001|  electronics| 599.99|       1|  2024-01-25 08:30:45|  Mobile|    USA|
|        TXN085|USER1085|   PROD085|home & garden|  365.5|       2|  2024-01-23 12:30:45|  Mobile|     UK|
|        TXN022|USER1022|   PROD022|  electronics| 899.99|       1|  2024-01-17 10:1

### 4. Partitioning & Performance
    Demonstrate awareness of Spark’s distributed architecture:

    Inspect and modify the number of partitions to optimize parallel processing.

    Use coalesce() or repartition() appropriately to reduce partitions before writing.

    Implement caching or persistence only for DataFrames reused in multiple actions.



In [44]:
# notes for partitions
'''
number of partitions = number of tasks
1 partition = 1 task 
and 1 task is assigned to one core

so if there are 4 cores
the tasks are executed in the batch of 4's (not exactly - as execution of each task and sparks scheduling affects this thing)


there should not be 1 partition only, it will lead to execution skewness

also there should not be too many partitions (around 200), it will lead to High scheduling overhead

ideal range of partitions ~= 2 or 4 * no. of cores

'''


try:
    print('step-4 started...')
    # inspecting number of partitions
    print(f'Number of partitions in products df before repartitioning: {valid_products_df.rdd.getNumPartitions()}')
    print(f'Number of partitions in transactions df before repartitioning: {valid_transactions_df.rdd.getNumPartitions()}')

    valid_products_df = valid_products_df.repartition(4)
    valid_transactions_df = valid_transactions_df.repartition(8)

    # caching
    valid_products_df.cache()
    valid_transactions_df.cache()

    print(f'Number of partitions in products df after repartitioning: {valid_products_df.rdd.getNumPartitions()}')
    print(f'Number of partitions in transactions df after repartitioning: {valid_transactions_df.rdd.getNumPartitions()}')

    print('step-4 finished successfully!')
except Exception as e: 
    print(f'Error in step-4(Data cleaning & standardization) {e}')
finally: 
    print('Step-4 over')

step-4 started...
Number of partitions in products df before repartitioning: 1
Number of partitions in transactions df before repartitioning: 200
Number of partitions in products df after repartitioning: 4
Number of partitions in transactions df after repartitioning: 8
step-4 finished successfully!
Step-4 over


### 5. Business Analytics & Aggregations
    Generate the following reports using distributed operations:

    Total Revenue: Calculated per country.

    Top Performers: Identify the top 5 products by total revenue.

    Platform Metrics: Average order value per platform.

    Trends: Daily transaction count trends.



In [45]:
try:
    print('Step-5 started...')

    # Total revenue
    print('Total revenue per country')
    total_revenue = valid_transactions_df\
                        .withColumn("revenue", col("quantity") * col("price"))\
                        .groupBy("country")\
                        .sum("revenue")\
                        .alias("total_revenue")
    total_revenue.show()

    # Top performers
    print('Top performers by total revenue')
    top_performers = valid_transactions_df\
                    .withColumn("revenue", col("quantity") * col("price"))\
                    .groupBy("product_id")\
                    .sum("revenue")\
                    .withColumnRenamed("sum(revenue)", "total_revenue")\
                    .orderBy(desc("total_revenue"))\
                    .limit(5)
    top_performers.show()

    # Platform metrics
    print("average order value per platform")
    platform_metrics = valid_transactions_df\
                        .withColumn("revenue", col("quantity") * col("price"))\
                        .groupBy("platform")\
                        .avg("revenue")\
                        .withColumnRenamed("avg(revenue)","platform_metrics")
                        
    platform_metrics.show()

    # Trends
    print("Trends: Daily transaction count")
    trends = valid_transactions_df\
                .withColumn("date",col("transaction_timestamp").cast("date"))\
                .groupBy("date")\
                .count()
    trends.show()

    print('Step-5 finished successfully')
except Exception as e: 
    print(f'Error in step-5(Business Analytics & Aggregations) {e}')
finally: 
    print('Step-5 over')

Step-5 started...
Total revenue per country
+-------+------------------+
|country|      sum(revenue)|
+-------+------------------+
| CANADA|          24937.34|
| FRANCE|          23598.71|
|    USA| 33045.78999999999|
|     UK|16451.729999999996|
|GERMANY|10498.780000000002|
|UNKNOWN|             299.0|
+-------+------------------+

Top performers by total revenue
+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|   PROD076|      3999.98|
|   PROD046|      3599.98|
|   PROD092|      3399.98|
|   PROD086|      3199.98|
|   PROD032|      3199.98|
+----------+-------------+

average order value per platform
+--------+-----------------+
|platform| platform_metrics|
+--------+-----------------+
|  Mobile|420.5980208333333|
|     Web|691.4539393939394|
+--------+-----------------+

Trends: Daily transaction count
+----------+-----+
|      date|count|
+----------+-----+
|2024-02-04|    2|
|2024-01-30|   10|
|2024-01-18|   10|
|2024-01-19|    9|
|2024-01-26|   1

### 6. Data Integration & Advanced Features
    Join Operations: Join transactional data with a reference dataset (Product name, Brand) using optimization techniques like Broadcast Joins.

    Tracking: Utilize Accumulators to track specific metrics (e.g., total invalid records) across the cluster.



In [46]:
from pyspark.sql.functions import broadcast

try:
    print('Step-6 started...')

    # accumulator for invalid products
    invalid_records_acc = spark.sparkContext.accumulator(0)

    invalid_products = valid_products_df.filter("product_id IS NULL")

    invalid_records_acc.add(invalid_products.count())

    # joining using broadcast join
    joined_df = valid_transactions_df.join(
        broadcast(valid_products_df)
        ,on=['product_id']
        ,how='inner'
    )

    joined_df = joined_df.select(
        'transaction_id',
        'product_id',
        'quantity',
        'price',
        'product_name',
        'brand'
    )

    joined_df.show()

    print('Invalid records accumulator value: ', invalid_records_acc.value)
    print('Step-6 finished successfully')
except Exception as e:
    print(f'Error in step-6(Data Integration & Advanced Features) {e}')
finally: 
    print('Step-6 over')

Step-6 started...
+--------------+----------+--------+-------+--------------------+-------------+
|transaction_id|product_id|quantity|  price|        product_name|        brand|
+--------------+----------+--------+-------+--------------------+-------------+
|        TXN023|   PROD023|       3|   75.0|         Winter Coat|    WarmStyle|
|        TXN066|   PROD066|       1|  625.0|     Graphics Tablet| DesignStudio|
|        TXN060|   PROD060|       3|  19.99|Science Fiction S...|  FutureTales|
|        TXN184|   PROD084|       1| 1125.0|       Monitor Stand| DeskOrganize|
|        TXN182|   PROD082|       1|  175.0|         Blouse Silk|   ElegantTop|
|        TXN162|   PROD062|       1|1399.99|          VR Headset| VirtualWorld|
|        TXN035|   PROD035|       1|  42.99|Self-Help Book Bu...|     MindGrow|
|        TXN118|   PROD018|       1| 199.99|        Evening Gown|  ElegantWear|
|        TXN107|   PROD007|       2|   45.0|Casual Cotton T-S...|    StyleWear|
|        TXN007|   PRO

### 7. Output Management
    Write the final processed dataset to the local filesystem.

    The output must use a columnar storage format.

    Partitioning: Organize the output folder by a logical business column.


In [47]:
try:
    print('Step-7 started...')
    df = valid_transactions_df

    df.write\
        .mode('overwrite')\
        .partitionBy('country')\
        .parquet('C:/tmp/output')
    
    print('Step-7 finished successfully')
except Exception as e:
    print(f'Error in step-7(Output Management) {e}')
finally: 
    print('Step-7 over')

Step-7 started...
Error in step-7(Output Management) An error occurred while calling o980.parquet.
: ExitCodeException exitCode=-1073741515: 
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1068)
	at org.apache.hadoop.util.Shell.run(Shell.java:959)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1282)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1377)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1359)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1116)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:798)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:838)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:810)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:837)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:810)